## **ROBERTA FOR NER**

This training pipeline exclude post-processing, compute-metrics at branch level and callbacks.

- Roberta model is expected to perform better than bert, especially with boundaries detection.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files

!cp /content/drive/MyDrive/src/evaluation.py /content/
!pip -q install datasets transformers seqeval scikit-learn pandas accelerate>=1.1.0

Mounted at /content/drive


In [7]:
# test imports
import evaluation

In [ ]:
import sys
sys.path.append("../notebooks")
sys.path.append("..")  # Add project root so 'src' is importable

### **1. IMPORT DEPENDENCIES**

In [8]:
import os
import csv
import time
from datetime import datetime

import torch
from datasets import load_from_disk
from evaluation import evaluate_resume_level_standard, save_to_dataframe

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

import warnings
warnings.filterwarnings("ignore")

### **2. CONFIG**

In [43]:
# For Colab use
DATASET_PATH = "/content/drive/MyDrive/resume_ner_hf_roberta-base"

MODEL_NAME = "roberta-base"

OUT_DIR = f"/content/checkpoints/{MODEL_NAME}"
LOG_DIR = f"/content/logs/{MODEL_NAME}"
RESULT_CSV = "/content/results/experiment_results.csv"
ENTITY_CSV = "/content/results/per_entity_results.csv"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

# ==================== MODEL HYPERPARAMETERS ====================
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
EPOCH = 3
WEIGHT_DECAY = 0.01
DROPOUT_RATE = 0.1
WARMUP_RATIO = 0.1
STRIDE = 128

In [ ]:
MODEL_NAME = "roberta-base"

# ==================== PATH ====================
DATASET_PATH = "../data/processed/resume_ner_hf_roberta-base"

OUT_DIR = f"../checkpoints/{MODEL_NAME}"
LOG_DIR = f"../logs/{MODEL_NAME}"
RESULT_CSV = "../results/experiment_results.csv"
ENTITY_CSV = "../results/per_entity_results.csv"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs("../results", exist_ok=True)

# ==================== MODEL HYPERPARAMETERS ====================
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
EPOCH = 4
WEIGHT_DECAY = 0.01
DROPOUT_RATE = 0.1
WARMUP_RATIO = 0.1
STRIDE = 128

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


### **3. LOAD DATASET**

In [11]:
ds = load_from_disk(DATASET_PATH)

# ==================== Label Setup ====================
# label should be manually define BIO tag
label_list = [
    "O",
    "B-JOB_TITLE",
    "I-JOB_TITLE",
    "B-SKILL",
    "I-SKILL",
    "B-EDUCATION",
    "I-EDUCATION",
]

id2label = {i: str(label) for i, label in enumerate(label_list)}
label2id = {str(label): i for i, label in enumerate(label_list)}

print("Data loaded successfully!")
print("Labels:", id2label)

Data loaded successfully!
Labels: {0: 'O', 1: 'B-JOB_TITLE', 2: 'I-JOB_TITLE', 3: 'B-SKILL', 4: 'I-SKILL', 5: 'B-EDUCATION', 6: 'I-EDUCATION'}


### **4. MODELLING**

In [44]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCH,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_dir=LOG_DIR,
    logging_strategy="epoch",

    fp16=torch.cuda.is_available(),
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=data_collator,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### **5. TRAINING & VALIDATION**

In [45]:
start_time = time.time()
trainer.train()
end_time = time.time()

training_time = end_time - start_time

print(f"Training Time: {training_time/60:.2f} minutes")

Epoch,Training Loss,Validation Loss
1,0.234053,0.126566
2,0.120869,0.120994
3,0.101782,0.121272


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Time: 11.40 minutes


In [46]:
val_results = evaluate_resume_level_standard(
    trainer=trainer,
    dataset=ds["validation"],
    id2label=id2label,
    stride=STRIDE,
)

print("\n===== Resume-Level Validation Results =====")
print(f"Precision: {val_results['precision']:.4f}")
print(f"Recall:    {val_results['recall']:.4f}")
print(f"F1:        {val_results['f1']:.4f}")


===== Resume-Level Validation Results =====
Precision: 0.6485
Recall:    0.5823
F1:        0.6136


In [47]:
val_df = save_to_dataframe(
    results=val_results,
    model_name=MODEL_NAME,
    split="validation",
    file_path=ENTITY_CSV
)

display(val_df)

,timestamp,model,split,entity,precision,recall,f1,support
0,2026-05-23 03:20:09,roberta-base,validation,EDUCATION,0.409910,0.452736,0.430260,201
1,2026-05-23 03:20:09,roberta-base,validation,JOB_TITLE,0.729417,0.692459,0.710457,2546
2,2026-05-23 03:20:09,roberta-base,validation,SKILL,0.633407,0.558565,0.593637,10706
3,2026-05-23 03:20:09,roberta-base,validation,micro avg,0.648510,0.582324,0.613637,13453
4,2026-05-23 03:20:09,roberta-base,validation,macro avg,0.590911,0.567920,0.578118,13453
5,2026-05-23 03:20:09,roberta-base,validation,weighted avg,0.648238,0.582324,0.613304,13453


### **6. TESTING**

In [48]:
test_results = evaluate_resume_level_standard(
    trainer=trainer,
    dataset=ds["test"],
    id2label=id2label,
    stride=STRIDE
)

print("\n===== Resume-Level Test Results =====")
print(f"Precision: {test_results['precision']:.4f}")
print(f"Recall:    {test_results['recall']:.4f}")
print(f"F1:        {test_results['f1']:.4f}")


===== Resume-Level Test Results =====
Precision: 0.6655
Recall:    0.6027
F1:        0.6326


In [49]:
test_df = save_to_dataframe(
    results=val_results,
    model_name=MODEL_NAME,
    split="test",
    file_path=ENTITY_CSV
)

display(test_df)

,timestamp,model,split,entity,precision,recall,f1,support
0,2026-05-23 03:20:34,roberta-base,test,EDUCATION,0.409910,0.452736,0.430260,201
1,2026-05-23 03:20:34,roberta-base,test,JOB_TITLE,0.729417,0.692459,0.710457,2546
2,2026-05-23 03:20:34,roberta-base,test,SKILL,0.633407,0.558565,0.593637,10706
3,2026-05-23 03:20:34,roberta-base,test,micro avg,0.648510,0.582324,0.613637,13453
4,2026-05-23 03:20:34,roberta-base,test,macro avg,0.590911,0.567920,0.578118,13453
5,2026-05-23 03:20:34,roberta-base,test,weighted avg,0.648238,0.582324,0.613304,13453


### **7. LOG RESULTS**

In [50]:
def append_row_to_csv(file_path, row):
    file_exists = os.path.exists(file_path)

    with open(file_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())

        if not file_exists:
            writer.writeheader()

        writer.writerow(row)

if device == "cuda":
    max_gpu_mem = torch.cuda.max_memory_allocated(0) / 1e9
else:
    max_gpu_mem = 0

log_data = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model":MODEL_NAME,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "validation_precision": val_results["precision"],
    "validation_recall": val_results["recall"],
    "validation_f1": val_results["f1"],
    "test_precision": test_results["precision"],
    "test_recall": test_results["recall"],
    "test_f1": test_results["f1"],
    "training_time_min": round(training_time / 60, 2),
    "max_gpu_memory_gb": round(max_gpu_mem, 2),
    "device": device
}

append_row_to_csv(RESULT_CSV, log_data)

print(f"\nExperiment logged to: {RESULT_CSV}")
print(f"Per-entity results logged to: {ENTITY_CSV}")


Experiment logged to: /content/results/experiment_results.csv
Per-entity results logged to: /content/results/per_entity_results.csv


## Model Export

Export the trained BERT NER model for use on different machines.
- `model.save_pretrained()` saves model weights and config in HuggingFace format.
- `tokenizer.save_pretrained()` saves the tokenizer vocab and settings.
- The exported directory can be loaded on any machine with `transformers` installed.


In [ ]:
import os
from pathlib import Path

# Export directory
EXPORT_DIR = Path("../exported_models/bert-base-uncased-ner")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save model and tokenizer in HuggingFace format (portable across machines)
model.save_pretrained(str(EXPORT_DIR))
tokenizer.save_pretrained(str(EXPORT_DIR))

# Save label mappings for inference
import json
label_config = {
    "id2label": id2label,
    "label2id": label2id,
    "label_list": label_list,
    "model_name": MODEL_NAME,
    "task": "ner"
}
with open(EXPORT_DIR / "label_config.json", "w") as f:
    json.dump(label_config, f, indent=2)

print(f"Model exported to: {EXPORT_DIR.resolve()}")
print(f"Files: {[f.name for f in EXPORT_DIR.iterdir()]}")
print()
print("=== Loading instructions ===")
print("from transformers import AutoTokenizer, AutoModelForTokenClassification")
print("import json")
print(f'tokenizer = AutoTokenizer.from_pretrained("{EXPORT_DIR}")')
print(f'model = AutoModelForTokenClassification.from_pretrained("{EXPORT_DIR}")')
print(f'with open("{EXPORT_DIR}/label_config.json") as f: label_cfg = json.load(f)')
print('id2label = {int(k): v for k, v in label_cfg["id2label"].items()}')


In [52]:
!cp /content/results/experiment_results.csv /content/drive/MyDrive/
!cp /content/results/per_entity_results.csv /content/drive/MyDrive/
print("Copied to MyDrive")

Copied to MyDrive
